# ABS Quarterly Private New Capital Expenditure 5625

## Python set-up

In [1]:
# analytic imports
import textwrap
import pandas as pd
import readabs as ra
from readabs import metacol as mc

In [2]:
# local imports
from abs_helper import get_abs_data
from mgplot import (
    multi_start,
    line_plot_finalise,
    series_growth_plot_finalise,
)

In [3]:
# pandas display settings
pd.options.display.max_rows = 999999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 999

# display charts
SHOW = False

## Get data from ABS

In [4]:
abs_dict, meta, source, _ = get_abs_data("5625.0")
plot_times = 0, -29

In [5]:
# A quick look at table names
textwrap.wrap(", ".join(abs_dict.keys()), width=80)

['01_current_prices_original_capex,',
 '02_current_prices_original_expected_short_capex,',
 '03_current_prices_original_expected_long_capex,',
 '04_current_prices_seasonally_adjusted_capex, 05_current_prices_trend_capex,',
 '06_volume_measures_original_capex, 07_volume_measures_seasonally_adjusted_capex,',
 '08_volume_measures_trend_capex, 09_current_prices_states_territories_capex,',
 '10_volume_measures_states_territories_capex,',
 '11_current_prices_new_south_wales_capex, 12_current_prices_victoria_capex,',
 '13_current_prices_queensland_capex, 14_current_prices_south_australia_capex,',
 '15_current_prices_western_australia_capex, 16_current_prices_tasmania_capex,',
 '17_current_prices_financial_year_capex,',
 '18_current_prices_realisation_ratios_capex,',
 '19_current_prices_mining_manufacturing_subdivisions_capex,',
 '20_current_prices_manufacturing_states_capex,',
 '21_current_prices_manufacturing_financial_year_realisation_ratios_capex']

In [6]:
# Map measure label -> (table name, price-label fragment used in the DID, short tag)
MEASURES = {
    "current prices": (
        "04_current_prices_seasonally_adjusted_capex",
        "Current Price",
        "CP",
    ),
    "chain volume measures": (
        "07_volume_measures_seasonally_adjusted_capex",
        "Chain Volume Measures",
        "CVM",
    ),
}
print("Latest data:", abs_dict[MEASURES["current prices"][0]].index[-1])

Latest data: 2026Q1


## Plot

In [7]:
def plot_level_and_growth(series, units, title, lfooter):
    """Plot the level ($) as a line chart and the Q/Y growth as a bar/line chart."""

    multi_start(
        series,
        function=line_plot_finalise,
        starts=plot_times,
        title=title,
        ylabel=units,
        rfooter=source,
        lfooter=lfooter,
        show=SHOW,
    )

    series_growth_plot_finalise(
        series,
        plot_from=-19,
        title=f"Growth: {title}",
        rfooter=source,
        lfooter=lfooter,
        show=SHOW,
    )


def get_capex(asset_type, industry, measure, series_type="Seasonally Adjusted"):
    """Fetch a capex series by asset type, industry and measure (CP or CVM)."""

    table, price_label, _short = MEASURES[measure]
    did = (
        f"Actual Expenditure ;  Total (State) ;  {asset_type} ;  "
        f"{price_label} ;  {industry} ;"
    )
    _t, sid, units = ra.find_abs_id(
        meta,
        {table: mc.table, series_type: mc.stype, did: mc.did},
        verbose=False,
    )
    series, units = ra.recalibrate(abs_dict[table][sid].dropna(), units)
    return series, units

### Headline: total capital expenditure

In [8]:
def headline():
    """Total private new capital expenditure: level and growth, CP and CVM."""

    series_type = "Seasonally Adjusted"
    for measure, (_table, _price, short) in MEASURES.items():
        series, units = get_capex(
            "Total (Type of Asset - Detailed Level)",
            "Total, including Education and Health",
            measure,
            series_type=series_type,
        )
        lfooter = (
            f"Australia. {series_type.capitalize()}. {measure.capitalize()}. "
        )
        title = f"CapEx: total: {short}"
        plot_level_and_growth(series, units, title, lfooter)


headline()

### By asset type

In [9]:
def by_asset_type():
    """Capex by asset type: Buildings & Structures vs Equipment, Plant & Machinery."""

    series_type = "Seasonally Adjusted"
    asset_types = ["Buildings and Structures", "Equipment, Plant and Machinery"]
    for asset in asset_types:
        for measure, (_table, _price, short) in MEASURES.items():
            series, units = get_capex(
                asset,
                "Total, including Education and Health",
                measure,
                series_type=series_type,
            )
            lfooter = (
                f"Australia. {series_type.capitalize()}. {measure.capitalize()}. "
            )
            title = f"CapEx: {asset}: {short}"
            plot_level_and_growth(series, units, title, lfooter)


by_asset_type()

### By industry sector

In [10]:
INDUSTRIES = [
    "Mining",
    "Non-Mining, including Education and Health",
    "Manufacturing",
    "Electricity, Gas, Water and Waste Services",
    "Construction",
    "Wholesale Trade",
    "Retail Trade",
    "Accommodation and Food Services",
    "Transport, Postal and Warehousing",
    "Information Media and Telecommunications",
    "Financial and Insurance Services",
    "Rental, Hiring and Real Estate Services",
    "Professional, Scientific and Technical Services",
    "Administrative and Support Services",
    "Education and Training",
    "Health Care and Social Assistance",
    "Arts and Recreation Services",
    "Other Services",
]


def by_industry():
    """Capex by industry sector, total asset type, in CP and CVM."""

    series_type = "Seasonally Adjusted"
    for industry in INDUSTRIES:
        for measure, (_table, _price, short) in MEASURES.items():
            series, units = get_capex(
                "Total (Type of Asset - Detailed Level)",
                industry,
                measure,
                series_type=series_type,
            )
            lfooter = (
                f"Australia. {series_type.capitalize()}. {measure.capitalize()}. "
            )
            title = f"CapEx: {industry}: {short}"
            plot_level_and_growth(series, units, title, lfooter)


by_industry()

### Data-centre / AI capex signature

The capex survey leads the national accounts and already covers the GDP quarter ahead of the published accounts. **Information Media & Telecommunications (IMT)** is where hyperscaler / data-centre investment is classified, and *equipment* within IMT (servers etc.) is the import-intensive leg.

1. **IMT equipment, full history**: flat at ~$0.3-0.5bn for 35 years, then a near-vertical ramp from 2023 (a sustained buildout, not a one-quarter spike).
2. **IMT equipment vs buildings**: the surge is hardware (equipment), not the data-centre shells (buildings), which is why it hits *imports*.

In [11]:
# --- Data-centre / AI capex signature -----------------------------------------
# The capex survey leads the national accounts and already covers the GDP quarter
# ahead of the published accounts. Information Media & Telecommunications (IMT) is
# where hyperscaler / data-centre investment is classified; equipment within IMT
# (servers etc.) is the import-intensive leg. Two views:
#   1) IMT equipment capex, full history (sustained ramp vs one-off spike);
#   2) IMT equipment vs buildings (hardware, not the data-centre shells).

_DC_INDUSTRY = "Information Media and Telecommunications"


def _capex_raw(asset, industry):
    """Raw (un-recalibrated) CVM SA capex level so two series share one scale."""
    table = MEASURES["chain volume measures"][0]
    did = (
        f"Actual Expenditure ;  Total (State) ;  {asset} ;  "
        f"Chain Volume Measures ;  {industry} ;"
    )
    _t, sid, _u = ra.find_abs_id(
        meta, {table: mc.table, "Seasonally Adjusted": mc.stype, did: mc.did},
        verbose=False,
    )
    return abs_dict[table][sid].dropna()


def datacentre_imt_equipment():
    """Chart 1: IMT equipment capex (servers etc.), level + growth, full history."""
    series, units = get_capex(
        "Equipment, Plant and Machinery", _DC_INDUSTRY, "chain volume measures",
    )
    plot_level_and_growth(
        series, units,
        title="CapEx: Info Media & Telecom: Equipment",
        lfooter="Australia. Seasonally Adjusted. Chain volume measures. Data-centre proxy. ",
    )


def datacentre_imt_equipment_vs_buildings():
    """Chart 2: IMT capex split into equipment (hardware) vs buildings (shells)."""
    df = pd.DataFrame({
        "Equipment (servers etc.)": _capex_raw("Equipment, Plant and Machinery", _DC_INDUSTRY),
        "Buildings & structures": _capex_raw("Buildings and Structures", _DC_INDUSTRY),
    }).dropna()
    line_plot_finalise(
        df,
        title="CapEx: Info Media & Telecom: Equipment vs Buildings",
        ylabel="$ million (CVM)",
        width=2,
        rfooter=source,
        lfooter="Australia. Seasonally Adjusted. Chain volume measures. Data-centre proxy. ",
        show=SHOW,
    )


datacentre_imt_equipment()
datacentre_imt_equipment_vs_buildings()


## Finished

In [12]:
# watermark
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-06-01 12:33:57

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.14.0

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.23
pandas : 3.0.3
readabs: 0.1.9

Watermark: 2.6.0



In [13]:
print("Finished")

Finished
